In [2]:
import os, json, yaml, time
import datetime
import urllib3
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Proj
from scipy.spatial import cKDTree
from itertools import product
from scipy.stats import bootstrap, ecdf
from shapely.geometry import Point, MultiPolygon
import multiprocessing
from functools import partial
from multiprocessing import Pool


### List and verify metrics files

In [3]:
base_data_path = "/glade/campaign/cisl/aiml/cat_flood/analysis_ready_data_20251121/"
start_date = "2023-03-15"
end_date = "2024-04-29"

date_range = pd.date_range(start=start_date, end=end_date, freq='d')
data_files = date_range.strftime(f"{base_data_path}veraset_%Y-%m-%d_PST_chunked.parquet")
files = []
for i, f in enumerate(data_files):
    if os.path.isfile(f):
        files.append(f)
print(len(files))

92


### Create boolean columns for FFWs and rain amounts during the AR period

#### FFWs

In [14]:
def get_bool(row, polys):
    point = Point(row[["lng_home", "lat_home"]])
    return any([p.contains(point) for p in polys])

In [ ]:
for file in files[58:64]:
    df_temp = pd.read_parquet(file)
    s_date = file.split("_PST")[0].split("veraset_")[1]
    e_date = (datetime.datetime.strptime(s_date, '%Y-%m-%d') + datetime.timedelta(days=1)).strftime('%Y-%m-%d')
    url = f"https://mesonet.agron.iastate.edu/api/1/cow.json?wfo=LOX&begints={s_date}T00:00:00Z&enable_shared_border=1&endts={e_date}T00:00:00Z&hailsize=1&wind=58&phenomena=FF&lsrtype=FF&lsrbuffer=30&warningbuffer=1"
    req = urllib3.request("GET", url)
    ffw_warnings = json.loads(req.data)["events"]["features"]
    polygons = [MultiPolygon(w["geometry"]["coordinates"]) for w in ffw_warnings]
    df_temp[f"FFW"] = df_temp.apply(get_bool, polys=polygons, axis=1)
    df_temp.to_parquet(file.replace("analysis_ready_data_20251121", "analysis_ready_data_20251121_metavars"))

In [ ]:
files_updated = [f.replace("analysis_ready_data_20251121", "analysis_ready_data_20251121_metavars") for f in files]

#### MRMS Variables

In [7]:
def create_tree(mrms_ds):
    # define projection
    proj = Proj(proj="lcc", R=6371229, lat_0=25, lon_0=265, lat_1=25, lat_2=25)
    
    # grid and project MRMS lat-lon coords then create KDTree
    mrms_lon_grid = np.tile(mrms_ds['lon'].values-360, (mrms_ds['lat'].values.shape[0], 1)).T
    mrms_lat_grid = np.tile(mrms_ds['lat'].values, (mrms_ds['lon'].values.shape[0], 1))
    proj_mrms_lon_grid, proj_mrms_lat_grid = np.array(proj(mrms_lon_grid, mrms_lat_grid))
    tree = cKDTree(np.c_[proj_mrms_lon_grid.ravel(), proj_mrms_lat_grid.ravel()])
    return tree, proj, mrms_lon_grid.shape

def get_mrms_var(df_temp, date, idx, mrms_ds, variable, variable_name):

    # grab MRMS data using calculated index
    lon_idx_da = xr.DataArray(idx[:, 0], dims="z")
    lat_idx_da = xr.DataArray(idx[:, 1], dims="z")
    mrms_idx = mrms_ds[variable].sel(time=date).isel(lon=lon_idx_da, lat=lat_idx_da).values
    series = pd.Series(mrms_idx.tolist(), name=variable_name, index=df_temp.index)
    return series

def cycle_through_mrms_variables(file, variables, variable_aggregates):

    mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variables[0]}_2024-03-15T00:00_2024-04-30T00:00_daily_{variable_aggregates[0]}.nc"
    mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
    tree, proj, mrms_lon_grid_shape = create_tree(mrms_ds)
    
    df_temp = pd.read_parquet(file)
    date = df_temp["date"].iloc[0].to_numpy()
    year = df_temp["date"].iloc[0].year
    _, points = tree.query(np.column_stack(proj(df_temp['lng_home'], df_temp['lat_home'])))
    idx = np.column_stack(np.unravel_index(points, mrms_lon_grid_shape))

    for variable, variable_aggregate in product(variables, variable_aggregates):
        mrms_fn = f"/glade/campaign/cisl/aiml/cat_flood/mrms/raw/{variable}_{year}-03-15T00:00_{year}-04-30T00:00_daily_{variable_aggregate}.nc"
        mrms_ds = xr.open_dataset(mrms_fn, engine="netcdf4")
        variable_name = f"streamflow_{variable_aggregate}" if "STREAMFLOW" in variable else f"precip_{variable_aggregate}"
        df_temp[f"{variable_name}"] = get_mrms_var(df_temp, date, idx, mrms_ds, variable, variable_name)
    df_temp.to_parquet(file)
    
    return None

In [9]:
variables = ["MultiSensor_QPE_01H_Pass2_00.00", "FLASH_HP_MAXUNITSTREAMFLOW_00.00"]
variable_aggregates = ["max", "sum"]

n_processors = multiprocessing.cpu_count()
f = partial(cycle_through_mrms_variables,
            variables=variables,
            variable_aggregates=variable_aggregates)

s = time.time()
print('Beginning data processing...')
with Pool(processes=n_processors) as pool:
    result = pool.map_async(f, files_updated[58:64])
    results = result.get()
print(f"{np.round((time.time() - s) / 60, 2)} minutes to complete parallel computation.")
print(len(results))

Beginning data processing...
0.23 minutes to complete parallel computation.
1


In [10]:
df_temp = pd.read_parquet(files_updated[58])
df_temp

,uid,duration,lat_home,lng_home,zipcode,row_group,county,number_of_visits,distance_straight_line,jump_lengths,radius_of_gyration,num_stops,max_distance_from_home,date,FFW,precip_max,precip_sum,streamflow_max,streamflow_sum
218,0000ba15f25930f0a4ad2bd7114a40bd30a14214b245a8...,3 days 13:28:52,33.929420,-118.051904,90605,0,LA,88,10.133705,"[0.0007038977772365412, 0.0, 0.000821825114659...",1.333580,10,3.259706,2023-03-15,False,7.3,46.9,1.9,20.0
248,0000a843eb75661d74cd9ba58f6830a1f84c5f49a0eac3...,0 days 00:23:52,33.927723,-118.203461,90262,0,LA,7,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0]",0.001408,1,0.000000,2023-03-15,False,7.9,51.9,1.0,13.1
260,0000a396879fd5767b3d146bfe8c854ea48c8e9a86b9f5...,4 days 18:17:15,34.162756,-119.168017,93033,0,VE,81,23.544740,"[0.22042789638748198, 0.34614753068159015, 0.0...",1.880407,5,6.822220,2023-03-15,False,7.5,35.2,1.2,13.5
475,0000712fa93eae897b02b840ce7a3be2b9eb8b06b12c46...,3 days 09:06:21,34.087652,-118.287289,90029,0,LA,15,1.356103,"[0.0022030198624645985, 0.0022030198624645985,...",0.158470,3,0.630850,2023-03-15,False,9.0,51.1,1.0,9.7
485,00006558db84477cda9278ba16d40cab37598b931ad2e1...,1 days 22:01:25,34.157461,-118.006886,91016,0,LA,2,0.001101,[0.001101141416629911],0.000653,1,0.001115,2023-03-15,False,11.6,70.7,1.9,16.3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10,fffff922ba116e91a1274046b2f7b158355940c5433bb7...,0 days 00:27:11,33.742100,-118.310204,90732,75017,LA,4,0.000000,"[0.0, 0.0, 0.0]",0.000000,1,0.000000,2023-03-15,False,7.3,56.1,2.5,18.1
37,ffffe79bdb634159d1e5d074c662e2336fe012548ed571...,1 days 22:53:37,34.394936,-118.908152,93015,75017,VE,16,33.708051,"[1.1559947293736152, 1.29116154228278, 1.14343...",3.359933,4,15.526305,2023-03-15,False,8.0,43.9,1.7,18.0
280,ffffc189a959d16f168dfb7f90ebf0355d8f1cfae1e797...,3 days 22:55:35,33.889900,-118.400803,90266,75017,LA,243,0.000000,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",0.001471,1,0.000000,2023-03-15,False,7.7,45.8,0.8,6.5
336,ffffacf186455724eff8a4c113f92378a24f2a6fa25d5d...,5 days 09:35:59,34.277335,-118.405703,91331,75017,LA,336,8.636476,"[0.0012725249472048672, 0.019297951426232113, ...",0.672377,1,3.176273,2023-03-15,False,7.1,40.6,2.5,19.3


In [ ]:
AR_dates = pd.date_range("2024-03-29", "2024-04-01", freq="1d")

bool_series = []
for date in AR_dates:
    df_temp = pd.read_parquet(files_updated[58].replace("2023-03-27", date.strftime('%Y-%m-%d')))
    bool_series.append(df_temp["FFW"].rename(f"FFW_{date.strftime('%Y-%m-%d')}"))
    for mrms_metric in ['precip_max', 'precip_sum', 'streamflow_max', 'streamflow_sum']:
        bool_series.append(df_temp.apply(lambda x: True if x[mrms_metric] > 0 else False, axis=1).rename(f"{mrms_metric}_{date.strftime('%Y-%m-%d')}"))
bool_df = pd.concat(bool_series, axis=1)

### Calculate Bootsrap average, sign, and p-value

In [ ]:
for mrms_label in ["precip", "streamflow"]:
    mrms_metrics = [c for c in list(df.columns) if any(k in c for k in [f'{mrms_label}_max_', f'{mrms_label}_sum_'])]
    mrms_metrics_bool = [c+b for c in mrms_metrics for b in ["_yes", "_no"]]
    mrms_metrics_bool_bs = [c+bs for c in mrms_metrics_bool for bs in ["_pvalue", "_boot_mean", "_boot_sign"]]
    deltas = [c for c in list(df.columns) if any(k in c for k in ['_z', '_modz'])]
    counties = ["LA", "SB", "VE"]
    bools = ["yes", "no"]
    
    for county in counties:
        df_bs = pd.DataFrame(columns=mrms_metrics_bool_bs, index=deltas)
        for mrms_metric in mrms_metrics:
            for delta in deltas:
                for bool in bools:
                    data = df.loc[(df[mrms_metric]==bool) & (df['county']==county)][delta].dropna()
                    if data.shape[0]<2:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = np.nan
                        continue
                    else:
                        res = bootstrap((data,), np.mean, confidence_level=0.95, paired=False, alternative='two-sided')
                        res_ecdf = ecdf(res.bootstrap_distribution)
                        res_ecdf_0 = float(res_ecdf.cdf.evaluate(0))
                        if res_ecdf_0 > 0.5:
                            df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = 1.0 - res_ecdf_0
                        else:
                            df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = res_ecdf_0
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_mean"] = np.mean(res.bootstrap_distribution)
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_sign"] = np.sign(np.mean(res.bootstrap_distribution))
        df_bs.to_parquet(f"bootstrapping_daily/results_{mrms_label}_{county}.parquet")

In [ ]:
mrms_metrics = [c for c in list(df.columns) if "_ffw" in c]
mrms_metrics_bool = [c+b for c in mrms_metrics for b in ["_yes", "_no"]]
mrms_metrics_bool_bs = [c+bs for c in mrms_metrics_bool for bs in ["_pvalue", "_boot_mean", "_boot_sign"]]
deltas = [c for c in list(df.columns) if any(k in c for k in ['_z', '_modz'])]
counties = ["LA", "SB", "VE"]
bools = ["yes", "no"]

for county in counties:
    df_bs = pd.DataFrame(columns=mrms_metrics_bool_bs, index=deltas)
    for mrms_metric in mrms_metrics:
        for delta in deltas:
            for bool in bools:
                data = df.loc[(df[mrms_metric]==bool) & (df['county']==county)][delta].dropna()
                if data.shape[0]<2:
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = np.nan
                    continue
                else:
                    res = bootstrap((data,), np.mean, confidence_level=0.95, paired=False, alternative='two-sided')
                    res_ecdf = ecdf(res.bootstrap_distribution)
                    res_ecdf_0 = float(res_ecdf.cdf.evaluate(0))
                    if res_ecdf_0 > 0.5:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = 1.0 - res_ecdf_0
                    else:
                        df_bs.loc[delta, f"{mrms_metric}_{bool}_pvalue"] = res_ecdf_0
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_mean"] = np.mean(res.bootstrap_distribution)
                    df_bs.loc[delta, f"{mrms_metric}_{bool}_boot_sign"] = np.sign(np.mean(res.bootstrap_distribution))
    df_bs.to_parquet(f"bootstrapping_daily/results_ffw_{county}.parquet")